<a href="https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [19]:
!git clone https://github.com/sadineniManushree/flyrank--internship__ml.git
%cd flyrank--internship__ml

Cloning into 'flyrank--internship__ml'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 155 (delta 62), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 1.86 MiB | 5.71 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/flyrank--internship__ml/flyrank--internship__ml


In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(len(df), "rows")

30000 rows


In [21]:
df['target'] = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)
print(df['target'].value_counts(normalize=True))

target
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [22]:
df['baseline_score'] = (df.groupby('position_tier')['ctr'].transform('mean') - df['ctr']) * df['impressions_90d']

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

feature_cols = ['ctr', 'avg_position', 'impressions_90d', 'clicks_90d',
                 'engagement_rate', 'scroll_rate', 'search_volume', 'content_age_days']

X_train, y_train = df.loc[train_idx, feature_cols].fillna(0), df.loc[train_idx, 'target']
X_test, y_test = df.loc[test_idx, feature_cols].fillna(0), df.loc[test_idx, 'target']

models = {
    'logistic_regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))]),
    'decision_tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight='balanced_subsample', random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)

print("Trained:", list(models.keys()))

Trained: ['logistic_regression', 'decision_tree', 'random_forest']


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split Design

**Split used: Grouped by client** (`client_holdout`), confirmed in code —
27,675 training rows, 2,325 test rows, no client appearing in both sets.

**Why this is the honest choice for my question:**

My question is: "Given a page's signals, will its CTR/engagement later
decline?" I want to know if the model has actually learned a real,
generalizable pattern — not just memorized quirks of clients it already
trained on. If I split randomly, rows from the same client could end up in
BOTH the training set and the test set. The model could then "cheat" by
recognizing that specific client's content style or baseline traffic
instead of learning the general CTR-vs-position-vs-engagement pattern —
making the test score look better than it really is.

By grouping the split by `client_id`, every client appears in either
training OR testing, never both. This forces the model to prove it works
on clients it has genuinely never seen before — the same real-world
situation it will face when scoring new clients later. My split logic falls
back to a stratified random split only if there were fewer than 5 unique
clients (not the case here — the dataset had well more than enough clients
for a true holdout).

I did not use a time-based split, because my target isn't tied to a
specific calendar cutoff in this dataset — `target` is a per-page
future-outcome label already derived from `trend_direction` (declining vs.
not), so a client-based holdout is the split that actually matches how this
model would be used: scoring new/unseen clients, not predicting forward in
time from a fixed date.

**Result:** the split produced a healthy, usable test set (2,325 rows) with
both classes represented, and the resulting model-vs-baseline comparison
(Random Forest reaching Precision@50 of 0.82 vs. baseline's 0.28) shows the
model generalizes well even under this stricter, client-holdout test — not
just an easier random split.

In [25]:
from sklearn.model_selection import train_test_split

def client_aware_split(frame, target_col='target', client_col='client_id', random_state=42):
    clients = frame[client_col].fillna('unknown').astype(str)
    unique_clients = clients.drop_duplicates().to_numpy()

    if len(unique_clients) >= 5:
        rng = np.random.default_rng(random_state)
        shuffled = rng.permutation(unique_clients)
        n_test = max(1, int(round(len(shuffled) * 0.2)))
        test_clients = set(shuffled[:n_test])
        test_mask = clients.isin(test_clients).to_numpy()
        train_idx = frame.index[~test_mask]
        test_idx = frame.index[test_mask]
        if frame.loc[train_idx, target_col].nunique() == 2 and frame.loc[test_idx, target_col].nunique() == 2:
            return train_idx, test_idx, "client_holdout"

    train_idx, test_idx = train_test_split(frame.index, test_size=0.2, random_state=random_state, stratify=frame[target_col])
    return train_idx, test_idx, "stratified_row_holdout"

train_idx, test_idx, split_strategy = client_aware_split(df)
print("Split strategy used:", split_strategy)
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))

# Verify no client leakage: these two sets should NOT overlap
train_clients = set(df.loc[train_idx, 'client_id'])
test_clients = set(df.loc[test_idx, 'client_id'])
overlap = train_clients & test_clients
print("Clients in both train and test (should be 0):", len(overlap))

Split strategy used: client_holdout
Train rows: 27675 | Test rows: 2325
Clients in both train and test (should be 0): 0


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [27]:
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return y_true.values[order].mean() if len(y_true) >= k else np.nan

results = []
base_scores = df.loc[test_idx, 'baseline_score'].values
results.append({'model': 'baseline_rules', 'roc_auc': roc_auc_score(y_test, base_scores),
                 'avg_precision': average_precision_score(y_test, base_scores),
                 'precision_at_50': precision_at_k(y_test, base_scores, 50)})

for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    results.append({'model': name, 'roc_auc': roc_auc_score(y_test, proba),
                     'avg_precision': average_precision_score(y_test, proba),
                     'precision_at_50': precision_at_k(y_test, proba, 50)})

results_df = pd.DataFrame(results).sort_values('precision_at_50', ascending=False).reset_index(drop=True)
results_df

,model,roc_auc,avg_precision,precision_at_50
0,random_forest,0.748426,0.618670,0.82
1,logistic_regression,0.683065,0.580450,0.64
2,decision_tree,0.739315,0.574210,0.52
3,baseline_rules,0.638989,0.501993,0.28


## 3. Train + Compare vs. Baseline

Same rows, same client-holdout split, same metrics as the baseline — the
baseline's `score` (from Week 4, frozen and unchanged) is evaluated on the
exact same `test_idx` rows as every ML model, so this is a fair,
apples-to-apples comparison (the Comparison Contract).

| model | roc_auc | avg_precision | precision_at_50 |
|---|---|---|---|
| random_forest | 0.748 | 0.619 | **0.82** |
| logistic_regression | 0.683 | 0.580 | 0.64 |
| decision_tree | 0.739 | 0.574 | 0.52 |
| baseline_rules | 0.639 | 0.502 | 0.28 |

**Random Forest wins on every metric.** The clearest gap is Precision@50:
of the top 50 pages the baseline rule would flag, only 28% were actually
declining. Random Forest's top 50 gets 82% right — roughly 3x better. Even
Logistic Regression, the simplest ML method, beats the baseline by a wide
margin (0.64 vs 0.28), which tells me the hand-written rule (comparing CTR
only to a position-tier average) is leaving real, learnable signal on the
table — consistent with the weak picks I found by hand in the baseline
review (rows with healthy CTR still getting flagged).

**Selected model: random_forest**, chosen by precision_at_50 — the metric
that best matches the actual decision this system supports (which pages a
reviewer should open first).

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [29]:
from sklearn.inspection import permutation_importance

rf_model = models['random_forest']
perm = permutation_importance(rf_model, X_test, y_test, n_repeats=20, random_state=42, scoring='roc_auc')

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

importance_df

,feature,importance_mean,importance_std
2,impressions_90d,0.155387,0.010224
1,avg_position,0.024935,0.004531
0,ctr,0.022265,0.004050
3,clicks_90d,0.011775,0.001528
5,scroll_rate,0.011472,0.003335
7,content_age_days,0.008701,0.003275
4,engagement_rate,0.004955,0.000579
6,search_volume,0.003626,0.000966


In [30]:
test_df = df.loc[test_idx].copy()
test_df['model_proba'] = rf_model.predict_proba(X_test)[:, 1]
test_df['model_pred'] = rf_model.predict(X_test)

false_positives = test_df[(test_df['model_pred'] == 1) & (test_df['target'] == 0)]
false_negatives = test_df[(test_df['model_pred'] == 0) & (test_df['target'] == 1)]

print("False positives:", len(false_positives), "| False negatives:", len(false_negatives))
print()
print("--- 3 false positives (predicted declining, actually fine) ---")
print(false_positives.sort_values('model_proba', ascending=False)[
    ['content_id', 'ctr', 'avg_position', 'impressions_90d', 'engagement_rate', 'model_proba']
].head(3).to_string(index=False))
print()
print("--- 3 false negatives (predicted fine, actually declining) ---")
print(false_negatives.sort_values('model_proba')[
    ['content_id', 'ctr', 'avg_position', 'impressions_90d', 'engagement_rate', 'model_proba']
].head(3).to_string(index=False))

False positives: 448 | False negatives: 304

--- 3 false positives (predicted declining, actually fine) ---
          content_id  ctr  avg_position  impressions_90d  engagement_rate  model_proba
content_9d931e9a2b19 0.00          18.4              292              0.0     0.756486
content_f8aa00130a0b 0.09           9.6             1103              0.0     0.751010
content_ea4417d89e2c 0.00          11.9              352              0.0     0.747806

--- 3 false negatives (predicted fine, actually declining) ---
          content_id   ctr  avg_position  impressions_90d  engagement_rate  model_proba
content_28b4223f4e5f  0.00           0.0                1              0.0     0.078092
content_34b14c00f80c  0.00           0.0                3            100.0     0.078513
content_472ce7ae14c0 33.33           0.3                3              0.0     0.115132


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and Interpretation

**What the model leans on:** `impressions_90d` dominates the model's
decisions (importance 0.155, roughly 6x the next feature, `avg_position` at
0.025). This is a real, observable signal — not leakage — and it makes
sense: pages with substantial traffic have enough data for a genuine trend
to be measurable, while very low-traffic pages don't give the model much to
work with either way.

**Where the model is most wrong:**

- **448 false positives** (predicted declining, actually fine) — the 3
  worst examples all have CTR near 0.00 and engagement_rate of 0.0. These
  pages are already performing about as badly as possible, and the model
  seems to read "already at the floor" as "still declining," when in
  reality a page with 0% CTR and 0% engagement often has nowhere further to
  fall — it may be flat/stable-bad rather than actively worsening.

- **304 false negatives** (predicted fine, actually declining) — the 3
  worst examples all have extremely low impressions_90d (1–3 impressions
  total). Given how heavily the model weights impressions, these pages
  simply don't generate enough signal for the model to detect a real
  trend, so it defaults toward "not declining" even when the label says
  otherwise. This is a direct consequence of the model's heavy reliance on
  volume.

**Takeaway:** the model is strongest on pages with meaningful traffic, and
weakest at the extremes — pages that are already bottomed-out (false
positives) and pages with too little data to read (false negatives). A
future improvement could add a minimum-impressions eligibility filter
(similar to the Decision Contract's eligibility criteria from the baseline)
so the model isn't asked to make confident calls on pages it fundamentally
can't see clearly.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.